# 네트워크 침입 탐지 — MLflow Pipeline v5

## 가설 H5: 전체 시간 순서에서 만든 과거 문맥 피처가 IDS 분류를 개선한다

v4는 `Timestamp`를 완전히 제외했습니다. v5는 Train/Test 전체 Parquet을 시간순으로 정렬하고,
현재 행보다 **엄격히 이전 시각**의 이벤트만 이용해 시간 문맥 피처를 생성합니다.

### 시간 피처

- 직전 고유 Timestamp와의 간격
- 과거 10초/60초/300초 이벤트 밀도(데이터셋 100만 행당 정규화)
- 과거 60초 `Flow Pkts/s`, `Flow Byts/s`, 전체 패킷 수 평균
- 시간대와 요일의 cyclic encoding
- 데이터 시작 이후 경과시간

### 검증 설계

전체 시간순 80/20 분할은 사용할 수 없습니다. 마지막 20%에는 `Benign`, `Infilteration`만 있고
앞 80%에는 `Infilteration`이 없기 때문입니다. 대신 각 클래스 내부를 Timestamp로 정렬하여 앞 80%를
학습, 뒤 20%를 검증에 사용합니다. 동일 Timestamp는 한쪽에만 배치합니다.

동일한 행과 분할에서 `no_temporal`과 `with_temporal`을 비교해 시간 피처 효과만 측정합니다.


In [1]:
from pathlib import Path
import json
import os
import time
import warnings

import duckdb
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

HYPOTHESIS_ID = "v5"
HYPOTHESIS = "past_only_temporal_context_features"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

WORK_ROOT = Path("/home/jovyan/work")
cwd = Path.cwd()
PROJECT_NAME = cwd.parent.name if cwd.name == "notebooks" else "network-classification"
PROJECT_DIR = WORK_ROOT / PROJECT_NAME

DATA_BASES = [WORK_ROOT / "datasets", Path("/home/jovyan/data")]
SUPPORTED_RAW = {".csv", ".xlsx", ".parquet"}

def has_train_file(raw_dir: Path) -> bool:
    return raw_dir.exists() and any(
        p.is_file()
        and p.suffix.lower() in SUPPORTED_RAW
        and p.stem.lower().startswith("train")
        for p in raw_dir.iterdir()
    )

preferred = [base / PROJECT_NAME for base in DATA_BASES]
DATA_ROOT = next((p for p in preferred if has_train_file(p / "raw")), None)
if DATA_ROOT is None:
    matches = []
    for base in DATA_BASES:
        if base.exists():
            matches.extend(
                raw_dir.parent for raw_dir in base.glob("*/raw")
                if has_train_file(raw_dir)
            )
    matches = list(dict.fromkeys(matches))
    if len(matches) != 1:
        raise RuntimeError(f"학습 데이터 프로젝트를 결정할 수 없습니다: {matches}")
    DATA_ROOT = matches[0]

RAW_DIR = DATA_ROOT / "raw"
INTERIM_DIR = DATA_ROOT / "interim"
PROCESSED_DIR = DATA_ROOT / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs" / HYPOTHESIS_ID
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET = INTERIM_DIR / "train.parquet"
TEST_PARQUET = INTERIM_DIR / "test.parquet"
TRAIN_TEMPORAL = PROCESSED_DIR / "train_temporal_v5.parquet"
TEST_TEMPORAL = PROCESSED_DIR / "test_temporal_v5.parquet"

ID_COL = "unique_id"
TARGET = "Label"
DROP_COLS = ["Timestamp"]
CAT_FEATURES = ["Dst Port", "Protocol"]
TRAIN_SAMPLE = 250_000
VALID_FRACTION = 0.20
INF_WEIGHT_MULTIPLIER = 0.50

CONTEXT_FEATURES = [
    "prev_distinct_ts_gap_s",
    "prior_events_10s_per_million",
    "prior_events_60s_per_million",
    "prior_events_300s_per_million",
    "prior_flow_pkts_s_mean_60s",
    "prior_flow_byts_s_mean_60s",
    "prior_total_pkts_mean_60s",
]
CALENDAR_FEATURES = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "elapsed_hours",
]
TEMPORAL_FEATURES = [*CONTEXT_FEATURES, *CALENDAR_FEATURES]

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
EXPERIMENT_NAME = f"{PROJECT_NAME}-ids-{HYPOTHESIS_ID}"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Notebook   :", cwd)
print("Project    :", PROJECT_NAME)
print("Data root  :", DATA_ROOT)
print("Processed  :", PROCESSED_DIR)
print("Output     :", OUTPUT_DIR)
print("Experiment :", EXPERIMENT_NAME)


Notebook   : /home/jovyan/work/network-classification/notebooks
Project    : network-classification
Data root  : /home/jovyan/work/datasets/ 📁 network-classification
Processed  : /home/jovyan/work/datasets/ 📁 network-classification/processed
Output     : /home/jovyan/work/network-classification/outputs/v5
Experiment : network-classification-ids-v5


## 1. Timestamp 구조와 전체 시간순 분할 불가 진단

In [2]:
def sql_path(path: Path) -> str:
    return str(path).replace("'", "''")

def parquet_expr(path: Path) -> str:
    return f"read_parquet('{sql_path(path)}')"

with duckdb.connect() as con:
    time_overview = con.execute(f'''
        SELECT
            COUNT(*) AS rows,
            MIN("Timestamp") AS min_ts,
            MAX("Timestamp") AS max_ts,
            COUNT(DISTINCT "Timestamp") AS distinct_ts,
            COUNT(*) - COUNT(DISTINCT "Timestamp") AS duplicate_ts_rows
        FROM {parquet_expr(TRAIN_PARQUET)}
    ''').df()
    chronological_split = con.execute(f'''
        WITH cutoff AS (
            SELECT quantile_cont("Timestamp", 0.8) AS ts
            FROM {parquet_expr(TRAIN_PARQUET)}
        )
        SELECT
            CASE WHEN t."Timestamp" < c.ts THEN 'early_80pct'
                 ELSE 'late_20pct' END AS split,
            t."{TARGET}" AS label,
            COUNT(*) AS count,
            MIN(t."Timestamp") AS min_ts,
            MAX(t."Timestamp") AS max_ts
        FROM {parquet_expr(TRAIN_PARQUET)} t
        CROSS JOIN cutoff c
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    ''').df()
    by_day = con.execute(f'''
        SELECT CAST("Timestamp" AS DATE) AS day_value,
               "{TARGET}" AS label, COUNT(*) AS count
        FROM {parquet_expr(TRAIN_PARQUET)}
        GROUP BY 1, 2 ORDER BY 1, 2
    ''').df()

display(time_overview)
display(chronological_split)
display(by_day)


,rows,min_ts,max_ts,distinct_ts,duplicate_ts_rows
0,4638804,2018-02-14 01:00:00,2018-03-02 12:59:59,222397,4416407


,split,label,count,min_ts,max_ts
0,early_80pct,Benign,2337726,2018-02-14 01:00:00,2018-02-23 12:20:57
1,early_80pct,DDOS,550194,2018-02-21 02:11:08,2018-02-21 10:43:16
2,early_80pct,DoS,518077,2018-02-15 09:27:42,2018-02-16 10:58:08
3,early_80pct,Brute Force,305045,2018-02-14 02:01:21,2018-02-23 11:02:58
4,late_20pct,Benign,854098,2018-02-23 12:20:58,2018-03-02 12:59:59
5,late_20pct,Infilteration,73664,2018-02-28 01:42:00,2018-03-01 10:54:59


,day_value,label,count
0,2018-02-14,Benign,441703
1,2018-02-14,Brute Force,304745
2,2018-02-15,Benign,646912
3,2018-02-15,DoS,42131
4,2018-02-16,Benign,72
5,2018-02-16,DoS,475946
6,2018-02-21,Benign,1411
7,2018-02-21,DDOS,550194
8,2018-02-22,Benign,648973
9,2018-02-23,Benign,643434


## 2. 전체 Parquet 시간 정렬 및 과거 전용 피처 생성

시간 집계는 222,397개의 고유 Timestamp 단위로 먼저 계산한 뒤 원본 행에 결합합니다.
윈도우의 끝을 현재 Timestamp보다 1마이크로초 이전으로 지정하므로 동일 시각의 peer 행은 포함하지 않습니다.


In [3]:
def build_temporal_parquet(source: Path, target: Path):
    feature_schema_version = 2
    version_path = target.with_suffix(target.suffix + ".version")
    if (
        target.exists()
        and version_path.exists()
        and version_path.read_text().strip() == str(feature_schema_version)
        and target.stat().st_mtime >= source.stat().st_mtime
    ):
        print("[재사용]", target)
        return target

    if target.exists():
        target.unlink()
        print("[교체] 이전 v5 가공 파일 제거:", target)

    print("[생성]", target)
    src = parquet_expr(source)
    dst = sql_path(target)
    query = f'''
        COPY (
            WITH per_ts AS (
                SELECT
                    "Timestamp" AS ts,
                    COUNT(*) AS n_events,
                    SUM(CASE WHEN isfinite("Flow Pkts/s")
                             THEN "Flow Pkts/s" ELSE 0 END) AS sum_flow_pkts_s,
                    SUM(CASE WHEN isfinite("Flow Byts/s")
                             THEN "Flow Byts/s" ELSE 0 END) AS sum_flow_byts_s,
                    SUM(COALESCE("Tot Fwd Pkts", 0) +
                        COALESCE("Tot Bwd Pkts", 0)) AS sum_total_pkts
                FROM {src}
                GROUP BY 1
            ),
            globals AS (
                SELECT COUNT(*)::DOUBLE AS total_rows,
                       MIN("Timestamp") AS min_ts
                FROM {src}
            ),
            context AS (
                SELECT
                    p.ts,
                    COALESCE(epoch(p.ts - LAG(p.ts) OVER (ORDER BY p.ts)), 0)
                        AS prev_distinct_ts_gap_s,
                    COALESCE(
                        1000000.0 * SUM(p.n_events) OVER w10 / g.total_rows, 0
                    ) AS prior_events_10s_per_million,
                    COALESCE(
                        1000000.0 * SUM(p.n_events) OVER w60 / g.total_rows, 0
                    ) AS prior_events_60s_per_million,
                    COALESCE(
                        1000000.0 * SUM(p.n_events) OVER w300 / g.total_rows, 0
                    ) AS prior_events_300s_per_million,
                    COALESCE(
                        SUM(p.sum_flow_pkts_s) OVER w60 /
                        NULLIF(SUM(p.n_events) OVER w60, 0), 0
                    ) AS prior_flow_pkts_s_mean_60s,
                    COALESCE(
                        SUM(p.sum_flow_byts_s) OVER w60 /
                        NULLIF(SUM(p.n_events) OVER w60, 0), 0
                    ) AS prior_flow_byts_s_mean_60s,
                    COALESCE(
                        SUM(p.sum_total_pkts) OVER w60 /
                        NULLIF(SUM(p.n_events) OVER w60, 0), 0
                    ) AS prior_total_pkts_mean_60s
                FROM per_ts p
                CROSS JOIN globals g
                WINDOW
                    w10 AS (ORDER BY p.ts RANGE BETWEEN
                        INTERVAL 10 SECOND PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING),
                    w60 AS (ORDER BY p.ts RANGE BETWEEN
                        INTERVAL 60 SECOND PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING),
                    w300 AS (ORDER BY p.ts RANGE BETWEEN
                        INTERVAL 300 SECOND PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING)
            )
            SELECT
                r.*,
                c.prev_distinct_ts_gap_s,
                c.prior_events_10s_per_million,
                c.prior_events_60s_per_million,
                c.prior_events_300s_per_million,
                c.prior_flow_pkts_s_mean_60s,
                c.prior_flow_byts_s_mean_60s,
                c.prior_total_pkts_mean_60s,
                sin(2 * pi() * date_part('hour', r."Timestamp") / 24.0) AS hour_sin,
                cos(2 * pi() * date_part('hour', r."Timestamp") / 24.0) AS hour_cos,
                sin(2 * pi() * date_part('dow', r."Timestamp") / 7.0) AS dow_sin,
                cos(2 * pi() * date_part('dow', r."Timestamp") / 7.0) AS dow_cos,
                epoch(r."Timestamp" - g.min_ts) / 3600.0 AS elapsed_hours
            FROM {src} r
            JOIN context c ON r."Timestamp" = c.ts
            CROSS JOIN globals g
            ORDER BY r."Timestamp", r."{ID_COL}"
        ) TO '{dst}' (FORMAT PARQUET, COMPRESSION ZSTD)
    '''
    started = time.time()
    with duckdb.connect() as con:
        con.execute(query)
    version_path.write_text(str(feature_schema_version))
    print(f"완료: {target} ({time.time() - started:,.1f}s)")
    return target

build_temporal_parquet(TRAIN_PARQUET, TRAIN_TEMPORAL)
build_temporal_parquet(TEST_PARQUET, TEST_TEMPORAL)


[교체] 이전 v5 가공 파일 제거: /home/jovyan/work/datasets/ 📁 network-classification/processed/train_temporal_v5.parquet
[생성] /home/jovyan/work/datasets/ 📁 network-classification/processed/train_temporal_v5.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

완료: /home/jovyan/work/datasets/ 📁 network-classification/processed/train_temporal_v5.parquet (6.0s)
[교체] 이전 v5 가공 파일 제거: /home/jovyan/work/datasets/ 📁 network-classification/processed/test_temporal_v5.parquet
[생성] /home/jovyan/work/datasets/ 📁 network-classification/processed/test_temporal_v5.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

완료: /home/jovyan/work/datasets/ 📁 network-classification/processed/test_temporal_v5.parquet (2.4s)


PosixPath('/home/jovyan/work/datasets/ 📁 network-classification/processed/test_temporal_v5.parquet')

In [4]:
def validate_temporal_parquet(source: Path, temporal: Path):
    with duckdb.connect() as con:
        return con.execute(f'''
            WITH ordered AS (
                SELECT "Timestamp",
                       LAG("Timestamp") OVER () AS previous_ts
                FROM {parquet_expr(temporal)}
            )
            SELECT
                (SELECT COUNT(*) FROM {parquet_expr(source)}) AS source_rows,
                (SELECT COUNT(*) FROM {parquet_expr(temporal)}) AS temporal_rows,
                COUNT(*) FILTER (WHERE "Timestamp" < previous_ts) AS order_violations,
                COUNT(*) FILTER (WHERE
                    prior_events_10s_per_million IS NULL OR
                    prior_events_60s_per_million IS NULL OR
                    prior_events_300s_per_million IS NULL OR
                    elapsed_hours IS NULL
                ) AS temporal_null_rows
            FROM ordered
            JOIN {parquet_expr(temporal)} USING ("Timestamp")
        ''').df()

# ORDER BY가 물리 Parquet 순서에 유지되는지 별도 positional scan으로 검증
with duckdb.connect() as con:
    train_check = con.execute(f'''
        WITH x AS (
            SELECT "Timestamp", LAG("Timestamp") OVER () AS previous_ts,
                   prior_events_10s_per_million,
                   prior_events_60s_per_million,
                   prior_events_300s_per_million,
                   prior_flow_pkts_s_mean_60s,
                   prior_flow_byts_s_mean_60s,
                   prior_total_pkts_mean_60s,
                   elapsed_hours
            FROM {parquet_expr(TRAIN_TEMPORAL)}
        )
        SELECT COUNT(*) AS rows,
               COUNT(*) FILTER (WHERE "Timestamp" < previous_ts) AS order_violations,
               COUNT(*) FILTER (WHERE NOT isfinite(prior_events_10s_per_million) OR
                                      NOT isfinite(prior_events_60s_per_million) OR
                                      NOT isfinite(prior_events_300s_per_million) OR
                                      NOT isfinite(prior_flow_pkts_s_mean_60s) OR
                                      NOT isfinite(prior_flow_byts_s_mean_60s) OR
                                      NOT isfinite(prior_total_pkts_mean_60s) OR
                                      NOT isfinite(elapsed_hours)) AS nonfinite_rows
        FROM x
    ''').df()
    test_check = con.execute(f'''
        WITH x AS (
            SELECT "Timestamp", LAG("Timestamp") OVER () AS previous_ts,
                   prior_events_10s_per_million,
                   prior_events_60s_per_million,
                   prior_events_300s_per_million,
                   prior_flow_pkts_s_mean_60s,
                   prior_flow_byts_s_mean_60s,
                   prior_total_pkts_mean_60s,
                   elapsed_hours
            FROM {parquet_expr(TEST_TEMPORAL)}
        )
        SELECT COUNT(*) AS rows,
               COUNT(*) FILTER (WHERE "Timestamp" < previous_ts) AS order_violations,
               COUNT(*) FILTER (WHERE NOT isfinite(prior_events_10s_per_million) OR
                                      NOT isfinite(prior_events_60s_per_million) OR
                                      NOT isfinite(prior_events_300s_per_million) OR
                                      NOT isfinite(prior_flow_pkts_s_mean_60s) OR
                                      NOT isfinite(prior_flow_byts_s_mean_60s) OR
                                      NOT isfinite(prior_total_pkts_mean_60s) OR
                                      NOT isfinite(elapsed_hours)) AS nonfinite_rows
        FROM x
    ''').df()

display(pd.concat({"train": train_check, "test": test_check}))
assert int(train_check.loc[0, "rows"]) == 4_638_804
assert int(test_check.loc[0, "rows"]) == 1_159_701
assert int(train_check.loc[0, "order_violations"]) == 0
assert int(test_check.loc[0, "order_violations"]) == 0
assert int(train_check.loc[0, "nonfinite_rows"]) == 0
assert int(test_check.loc[0, "nonfinite_rows"]) == 0


,,rows,order_violations,nonfinite_rows
train,0,4638804,0,0
test,0,1159701,0,0


## 3. 전체 시간 피처 생성 후 분석 샘플 로딩

In [5]:
def load_sample(path: Path, n_rows: int):
    with duckdb.connect() as con:
        return con.execute(f'''
            SELECT * FROM {parquet_expr(path)}
            USING SAMPLE reservoir({int(n_rows)} ROWS)
            REPEATABLE ({RANDOM_STATE})
        ''').df()

train_df = load_sample(TRAIN_TEMPORAL, TRAIN_SAMPLE)
train_df["Timestamp"] = pd.to_datetime(train_df["Timestamp"])
print("Loaded:", train_df.shape)
print(f"Memory: {train_df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
display(train_df[[TARGET, "Timestamp", *TEMPORAL_FEATURES]].head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded: (250000, 55)
Memory: 137.8 MB


,Label,Timestamp,prev_distinct_ts_gap_s,prior_events_10s_per_million,prior_events_60s_per_million,prior_events_300s_per_million,prior_flow_pkts_s_mean_60s,prior_flow_byts_s_mean_60s,prior_total_pkts_mean_60s,hour_sin,hour_cos,dow_sin,dow_cos,elapsed_hours
0,Benign,2018-02-14 08:30:58,1.0,216.435098,831.248744,1379.234820,2727.636322,122474.777518,12.835322,8.660254e-01,-0.500000,0.433884,-0.900969,7.516111
1,DDOS,2018-02-21 02:16:30,1.0,1375.354509,8264.845853,41364.325805,1374.591548,190359.560468,4.491067,5.000000e-01,0.866025,0.433884,-0.900969,169.275000
2,Benign,2018-02-28 02:40:58,1.0,19.617125,149.176383,411.744062,2198.256618,104510.935592,17.248555,5.000000e-01,0.866025,0.433884,-0.900969,337.682778
3,Benign,2018-02-23 12:15:22,1.0,28.240038,398.378548,1470.853263,4533.443730,153886.972685,12.214286,1.224647e-16,-1.000000,-0.974928,-0.222521,227.256111
4,Benign,2018-03-01 04:01:53,1.0,62.947260,81.702094,437.612798,1985.238248,98632.538395,12.290237,8.660254e-01,0.500000,-0.433884,-0.900969,363.031389


## 4. v4 IDS 피처와 클래스별 시간 블록 분할

In [6]:
def add_ids_features(df: pd.DataFrame):
    df = df.copy()
    flag_cols = [
        "SYN Flag Cnt", "ACK Flag Cnt", "RST Flag Cnt",
        "FIN Flag Cnt", "URG Flag Cnt", "PSH Flag Cnt",
    ]
    if all(c in df.columns for c in flag_cols):
        values = {
            c: pd.to_numeric(df[c], errors="coerce").fillna(0).astype(float)
            for c in flag_cols
        }
        syn, ack = values["SYN Flag Cnt"], values["ACK Flag Cnt"]
        rst, fin = values["RST Flag Cnt"], values["FIN Flag Cnt"]
        urg, psh = values["URG Flag Cnt"], values["PSH Flag Cnt"]
        total_flags = syn + ack + rst + fin + urg + psh + 1.0
        df["SYN_ACK_ratio"] = (syn + 1) / (ack + 1)
        df["RST_ACK_ratio"] = (rst + 1) / (ack + 1)
        df["SYN_RST_over_ACK_FIN"] = (syn + rst + 1) / (ack + fin + 1)
        df["ACK_SYN_ratio"] = (ack + 1) / (syn + 1)
        df["URG_ACK_ratio"] = (urg + 1) / (ack + 1)
        df["SYN_share"] = (syn + 1) / total_flags
        df["RST_share"] = (rst + 1) / total_flags
    numeric = df.select_dtypes(include=[np.number]).columns
    df[numeric] = df[numeric].replace([np.inf, -np.inf], np.nan)
    return df

train_df = add_ids_features(train_df)
classes = sorted(train_df[TARGET].astype(str).unique().tolist())
attack_classes = [c for c in classes if c != "Benign"]
INFILTRATION_LABEL = next(
    c for c in classes
    if c.strip().lower() in {"infiltration", "infilteration"}
)

# Timestamp tie를 쪼개지 않도록 클래스별 80% quantile 시각을 경계로 사용
class_cutoffs = train_df.groupby(TARGET)["Timestamp"].quantile(1 - VALID_FRACTION)
cutoff_per_row = train_df[TARGET].map(class_cutoffs)
valid_mask = train_df["Timestamp"] > cutoff_per_row
train_mask = ~valid_mask

split_distribution = pd.crosstab(
    pd.Series(np.where(valid_mask, "valid_late", "train_early"), index=train_df.index),
    train_df[TARGET],
)
display(class_cutoffs.to_frame("cutoff"))
display(split_distribution)
assert (split_distribution > 0).all().all()

y_all = train_df[TARGET].astype(str)
y_train = y_all.loc[train_mask]
y_valid = y_all.loc[valid_mask]
print("Train rows:", len(y_train), "Valid rows:", len(y_valid))


,cutoff
Label,
Benign,2018-02-28 11:15:54.000
Brute Force,2018-02-14 11:32:57.200
DDOS,2018-02-21 02:27:14.000
DoS,2018-02-16 10:13:27.000
Infilteration,2018-03-01 03:34:09.000


Label,Benign,Brute Force,DDOS,DoS,Infilteration
row_0,,,,,
train_early,137495,13146,23894,22345,3132
valid_late,34373,3287,5965,5586,777


Train rows: 200012 Valid rows: 49988


## 5. 동일 분할에서 시간 피처 유무 비교

In [7]:
drop_common = [TARGET, ID_COL, *DROP_COLS]
X_temporal_all = train_df.drop(columns=drop_common, errors="ignore")
X_base_all = X_temporal_all.drop(columns=TEMPORAL_FEATURES, errors="ignore")
X_context_all = X_temporal_all.drop(columns=CALENDAR_FEATURES, errors="ignore")

for frame in [X_base_all, X_context_all, X_temporal_all]:
    for column in CAT_FEATURES:
        if column in frame.columns:
            frame[column] = frame[column].astype("string")

X_sets = {
    "no_temporal": X_base_all,
    "past_context_only": X_context_all,
    "all_temporal": X_temporal_all,
}

balanced_values = compute_class_weight(
    class_weight="balanced", classes=np.asarray(classes), y=y_train.to_numpy()
)
class_weights = dict(zip(classes, balanced_values.astype(float)))
class_weights[INFILTRATION_LABEL] *= INF_WEIGHT_MULTIPLIER
display(pd.Series(class_weights, name="class_weight").to_frame())

def make_preprocessor(frame):
    cat_features = [c for c in CAT_FEATURES if c in frame.columns]
    num_features = [c for c in frame.columns if c not in cat_features]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features),
    ])

def evaluate(y_true, y_pred):
    metrics = {
        "weighted_f1_attacks": float(f1_score(
            y_true, y_pred, labels=attack_classes,
            average="weighted", zero_division=0,
        )),
        "macro_f1_attacks": float(f1_score(
            y_true, y_pred, labels=attack_classes,
            average="macro", zero_division=0,
        )),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[INFILTRATION_LABEL],
        average=None, zero_division=0,
    )
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    metrics.update({
        "infilteration_precision": float(p[0]),
        "infilteration_recall": float(r[0]),
        "infilteration_f1": float(f[0]),
        "benign_to_infilteration_fp": int(np.sum(
            (yt == "Benign") & (yp == INFILTRATION_LABEL)
        )),
    })
    return metrics


,class_weight
Benign,0.290937
Brute Force,3.042933
DDOS,1.674161
DoS,1.790217
Infilteration,6.386079


In [8]:
def run_experiment(feature_set, frame):
    X_train = frame.loc[train_mask]
    X_valid = frame.loc[valid_mask]
    pipeline = Pipeline([
        ("preprocess", make_preprocessor(frame)),
        ("model", LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            max_depth=-1, subsample=0.9, colsample_bytree=0.9,
            objective="multiclass", class_weight=class_weights,
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
        )),
    ])
    run_name = f"{HYPOTHESIS_ID}_{feature_set}"
    started = time.time()
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "project": PROJECT_NAME,
            "hypothesis_id": HYPOTHESIS_ID,
            "hypothesis": HYPOTHESIS,
            "feature_set": feature_set,
            "validation": "within_class_chronological_80_20",
        })
        mlflow.log_params({
            "feature_set": feature_set,
            "train_sample": TRAIN_SAMPLE,
            "valid_fraction": VALID_FRACTION,
            "n_features": frame.shape[1],
            "inf_weight_multiplier": INF_WEIGHT_MULTIPLIER,
            "class_weight_json": json.dumps(class_weights, sort_keys=True),
            "past_only_windows": "10s,60s,300s",
        })
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_valid)
        metrics = evaluate(y_valid, pred)
        metrics["train_seconds"] = time.time() - started
        mlflow.log_metrics(metrics)

        safe = run_name.lower()
        report_path = OUTPUT_DIR / f"{safe}_classification_report.csv"
        pd.DataFrame(classification_report(
            y_valid, pred, labels=classes,
            output_dict=True, zero_division=0,
        )).T.to_csv(report_path, encoding="utf-8-sig")
        cm = confusion_matrix(y_valid, pred, labels=classes)
        fig, ax = plt.subplots(figsize=(8, 7))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=classes, yticklabels=classes, ax=ax)
        ax.set(title=f"Confusion Matrix — {run_name}",
               xlabel="Predicted", ylabel="Actual")
        plt.tight_layout()
        cm_path = OUTPUT_DIR / f"{safe}_confusion_matrix.png"
        fig.savefig(cm_path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(str(report_path), artifact_path="evaluation")
        mlflow.log_artifact(str(cm_path), artifact_path="evaluation")
        mlflow.sklearn.log_model(
            pipeline, name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        run_id = run.info.run_id
    print(run_name, metrics, "run_id=", run_id)
    return {
        "feature_set": feature_set,
        "pipeline": pipeline,
        "metrics": metrics,
        "run_id": run_id,
        "pred": pred,
    }

results = [run_experiment(name, frame) for name, frame in X_sets.items()]


2026/08/15 15:33:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v5_no_temporal at: http://mlflow:5000/#/experiments/4/runs/c682f02447f049c28940313b683583cf
🧪 View experiment at: http://mlflow:5000/#/experiments/4
v5_no_temporal {'weighted_f1_attacks': 0.6495909284899782, 'macro_f1_attacks': 0.5198926406457288, 'accuracy': 0.6401536368728494, 'infilteration_precision': 0.03907412920512057, 'infilteration_recall': 0.6756756756756757, 'infilteration_f1': 0.07387602898754661, 'benign_to_infilteration_fp': 12906, 'train_seconds': 24.22174859046936} run_id= c682f02447f049c28940313b683583cf


2026/08/15 15:33:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v5_past_context_only at: http://mlflow:5000/#/experiments/4/runs/068c37977c5a4afe9b679645d008aea5
🧪 View experiment at: http://mlflow:5000/#/experiments/4
v5_past_context_only {'weighted_f1_attacks': 0.7115374338053269, 'macro_f1_attacks': 0.5781985194330038, 'accuracy': 0.8369208610066416, 'infilteration_precision': 0.08560572194903888, 'infilteration_recall': 0.4929214929214929, 'infilteration_f1': 0.14587697581413064, 'benign_to_infilteration_fp': 4091, 'train_seconds': 25.8814480304718} run_id= 068c37977c5a4afe9b679645d008aea5


2026/08/15 15:34:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v5_all_temporal at: http://mlflow:5000/#/experiments/4/runs/cc3edbda3c244cbcb42bd565f23dbec2
🧪 View experiment at: http://mlflow:5000/#/experiments/4
v5_all_temporal {'weighted_f1_attacks': 0.9364266607275201, 'macro_f1_attacks': 0.7430500663753623, 'accuracy': 0.3040729775146035, 'infilteration_precision': 0.022141167754252986, 'infilteration_recall': 1.0, 'infilteration_f1': 0.04332311123501533, 'benign_to_infilteration_fp': 34316, 'train_seconds': 24.243263006210327} run_id= cc3edbda3c244cbcb42bd565f23dbec2


## 6. 시간 피처 효과 및 중요도

In [9]:
summary = pd.DataFrame([
    {"feature_set": r["feature_set"], **r["metrics"], "run_id": r["run_id"]}
    for r in results
])
summary_path = OUTPUT_DIR / "v5_temporal_comparison.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
display(summary)

base_result = next(r for r in results if r["feature_set"] == "no_temporal")
temporal_results = [r for r in results if r["feature_set"] != "no_temporal"]
for result in temporal_results:
    deltas = {
        key: result["metrics"][key] - base_result["metrics"][key]
        for key in [
            "weighted_f1_attacks", "macro_f1_attacks", "accuracy",
            "infilteration_precision", "infilteration_recall", "infilteration_f1",
            "benign_to_infilteration_fp",
        ]
    }
    print(result["feature_set"], "deltas:", deltas)

# 공격 F1 단일 지표가 정상 오탐 붕괴를 숨기지 않도록 운영 안전조건 적용
eligible_temporal = [
    r for r in temporal_results
    if r["metrics"]["accuracy"] >= base_result["metrics"]["accuracy"] - 0.02
    and r["metrics"]["infilteration_f1"] >= base_result["metrics"]["infilteration_f1"] - 0.01
]
selection_pool = [base_result, *eligible_temporal]
best_result = max(selection_pool, key=lambda r: r["metrics"]["weighted_f1_attacks"])
print("Champion:", best_result["feature_set"])
print("Eligible temporal feature sets:", [r["feature_set"] for r in eligible_temporal])


,feature_set,weighted_f1_attacks,macro_f1_attacks,accuracy,infilteration_precision,infilteration_recall,infilteration_f1,benign_to_infilteration_fp,train_seconds,run_id
0,no_temporal,0.649591,0.519893,0.640154,0.039074,0.675676,0.073876,12906,24.221749,c682f02447f049c28940313b683583cf
1,past_context_only,0.711537,0.578199,0.836921,0.085606,0.492921,0.145877,4091,25.881448,068c37977c5a4afe9b679645d008aea5
2,all_temporal,0.936427,0.743050,0.304073,0.022141,1.000000,0.043323,34316,24.243263,cc3edbda3c244cbcb42bd565f23dbec2


past_context_only deltas: {'weighted_f1_attacks': 0.06194650531534862, 'macro_f1_attacks': 0.058305878787275, 'accuracy': 0.19676722413379222, 'infilteration_precision': 0.04653159274391831, 'infilteration_recall': -0.18275418275418276, 'infilteration_f1': 0.07200094682658403, 'benign_to_infilteration_fp': -8815}
all_temporal deltas: {'weighted_f1_attacks': 0.2868357322375419, 'macro_f1_attacks': 0.22315742572963349, 'accuracy': -0.33608065935824594, 'infilteration_precision': -0.016932961450867586, 'infilteration_recall': 0.32432432432432434, 'infilteration_f1': -0.03055291775253128, 'benign_to_infilteration_fp': 21410}
Champion: past_context_only
Eligible temporal feature sets: ['past_context_only']


In [10]:
all_temporal_result = next(
    r for r in results if r["feature_set"] == "all_temporal"
)
temporal_model = all_temporal_result["pipeline"].named_steps["model"]
temporal_preprocessor = all_temporal_result["pipeline"].named_steps["preprocess"]
feature_importance = pd.DataFrame({
    "feature": temporal_preprocessor.get_feature_names_out(),
    "gain": temporal_model.booster_.feature_importance(importance_type="gain"),
    "split": temporal_model.booster_.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)
feature_importance["gain_ratio"] = feature_importance["gain"] / feature_importance["gain"].sum()
importance_path = OUTPUT_DIR / "v5_temporal_feature_importance.csv"
feature_importance.to_csv(importance_path, index=False, encoding="utf-8-sig")
display(feature_importance.head(30))

temporal_importance = feature_importance[
    feature_importance["feature"].str.replace("num__", "", regex=False).isin(TEMPORAL_FEATURES)
]
display(temporal_importance)


,feature,gain,split,gain_ratio
42,num__prior_flow_pkts_s_mean_60s,1.817014e+06,6454,0.309379
49,num__elapsed_hours,1.192493e+06,6995,0.203044
39,num__prior_events_10s_per_million,1.131155e+06,5828,0.192600
41,num__prior_events_300s_per_million,8.723746e+05,6609,0.148538
44,num__prior_total_pkts_mean_60s,1.246453e+05,6930,0.021223
46,num__hour_cos,8.721690e+04,896,0.014850
7,num__Flow IAT Mean,7.768755e+04,3270,0.013228
45,num__hour_sin,7.059521e+04,1425,0.012020
55,num__SYN_share,6.578560e+04,588,0.011201
10,num__Flow IAT Min,6.385849e+04,6590,0.010873


,feature,gain,split,gain_ratio
42,num__prior_flow_pkts_s_mean_60s,1.817014e+06,6454,3.093794e-01
49,num__elapsed_hours,1.192493e+06,6995,2.030435e-01
39,num__prior_events_10s_per_million,1.131155e+06,5828,1.925996e-01
41,num__prior_events_300s_per_million,8.723746e+05,6609,1.485376e-01
44,num__prior_total_pkts_mean_60s,1.246453e+05,6930,2.122312e-02
46,num__hour_cos,8.721690e+04,896,1.485025e-02
45,num__hour_sin,7.059521e+04,1425,1.202011e-02
47,num__dow_sin,6.340455e+04,669,1.079577e-02
40,num__prior_events_60s_per_million,5.667194e+04,4544,9.649422e-03
43,num__prior_flow_byts_s_mean_60s,3.175972e+04,5367,5.407666e-03


## 7. Champion 전체 샘플 재학습 및 Test 예측

In [11]:
champion_feature_set = best_result["feature_set"]
champion_frame = X_sets[champion_feature_set]
final_pipeline = clone(best_result["pipeline"])

with mlflow.start_run(run_name=f"{HYPOTHESIS_ID}_champion_full_refit") as final_run:
    mlflow.set_tags({
        "project": PROJECT_NAME,
        "hypothesis_id": HYPOTHESIS_ID,
        "hypothesis": HYPOTHESIS,
        "stage": "full_sample_refit",
        "selected_feature_set": champion_feature_set,
    })
    mlflow.log_params({
        "selected_feature_set": champion_feature_set,
        "train_rows": len(champion_frame),
        "source_validation_run_id": best_result["run_id"],
    })
    started = time.time()
    final_pipeline.fit(champion_frame, y_all)
    mlflow.log_metric("train_seconds", time.time() - started)
    mlflow.sklearn.log_model(
        final_pipeline, name="model",
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )
    final_run_id = final_run.info.run_id

print("Full refit run_id:", final_run_id)


2026/08/15 15:34:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v5_champion_full_refit at: http://mlflow:5000/#/experiments/4/runs/15dbb6b3570c4869a0f97c566830078c
🧪 View experiment at: http://mlflow:5000/#/experiments/4
Full refit run_id: 15dbb6b3570c4869a0f97c566830078c


In [12]:
def load_full(path: Path):
    with duckdb.connect() as con:
        return con.execute(f"SELECT * FROM {parquet_expr(path)}").df()

test_df = add_ids_features(load_full(TEST_TEMPORAL))
test_ids = test_df[ID_COL].copy()
X_test_temporal = test_df.drop(columns=[TARGET, ID_COL, *DROP_COLS], errors="ignore")
for column in CAT_FEATURES:
    if column in X_test_temporal.columns:
        X_test_temporal[column] = X_test_temporal[column].astype("string")
X_test_base = X_test_temporal.drop(columns=TEMPORAL_FEATURES, errors="ignore")
X_test_context = X_test_temporal.drop(columns=CALENDAR_FEATURES, errors="ignore")
X_test_sets = {
    "no_temporal": X_test_base,
    "past_context_only": X_test_context,
    "all_temporal": X_test_temporal,
}
X_test = X_test_sets[champion_feature_set]

expected_columns = champion_frame.columns.tolist()
if set(X_test.columns) != set(expected_columns):
    raise ValueError(
        f"Train/Test mismatch: missing={set(expected_columns)-set(X_test.columns)}, "
        f"extra={set(X_test.columns)-set(expected_columns)}"
    )
X_test = X_test[expected_columns]
test_pred = final_pipeline.predict(X_test)
submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_pred}).set_index(ID_COL)
submission_path = OUTPUT_DIR / "submission_v5.csv"
submission.to_csv(submission_path, encoding="utf-8-sig")
print("Saved:", submission_path, submission.shape)
display(submission[TARGET].value_counts().to_frame("count"))


Saved: /home/jovyan/work/network-classification/outputs/v5/submission_v5.csv (1159701, 1)


,count
Label,
Benign,796396
DDOS,136877
DoS,129046
Brute Force,76201
Infilteration,21181


## 8. 산출물 검증과 가설 판정

시간 피처가 성능을 높이더라도 날짜별 공격 배치가 강하므로, 이는 새로운 날짜의 공격 일반화 성능이 아니라
이 데이터셋 내부의 시간 스케줄 신호를 활용한 결과로 해석해야 합니다.


In [13]:
with duckdb.connect() as con:
    test_rows = con.execute(
        f"SELECT COUNT(*) FROM {parquet_expr(TEST_TEMPORAL)}"
    ).fetchone()[0]

checks = {
    "rows": len(submission),
    "expected_rows": test_rows,
    "label_missing": int(submission[TARGET].isna().sum()),
    "duplicate_id": int(submission.index.duplicated().sum()),
    "unknown_labels": sorted(set(submission[TARGET]) - set(classes)),
}
print("Submission checks:", checks)
assert checks["rows"] == checks["expected_rows"]
assert checks["label_missing"] == 0
assert checks["duplicate_id"] == 0
assert not checks["unknown_labels"]

hypothesis_supported = (
    best_result["feature_set"] != "no_temporal"
    and best_result["metrics"]["weighted_f1_attacks"]
        > base_result["metrics"]["weighted_f1_attacks"]
)
print("Hypothesis supported:", hypothesis_supported)
print("Summary CSV         :", summary_path)
print("Importance CSV      :", importance_path)
print("Final MLflow run    :", final_run_id)


Submission checks: {'rows': 1159701, 'expected_rows': 1159701, 'label_missing': 0, 'duplicate_id': 0, 'unknown_labels': []}
Hypothesis supported: True
Summary CSV         : /home/jovyan/work/network-classification/outputs/v5/v5_temporal_comparison.csv
Importance CSV      : /home/jovyan/work/network-classification/outputs/v5/v5_temporal_feature_importance.csv
Final MLflow run    : 15dbb6b3570c4869a0f97c566830078c
